# Exhaustive analysis of antibiotic resistance correlations

This notebook analyzes pairwise resistance relationships between antibiotics in the MALDI-AMR dataset.

Main outputs:

- Global antibiotic-antibiotic correlation matrix.
- Per-species correlation matrices.
- Top positive and negative resistance correlations.
- Co-resistance conditional probabilities: `P(B resistant | A resistant)`.
- Mutual information between antibiotic resistance labels.
- Optional clustered heatmaps.
- Exported CSV files for downstream analysis.

The analysis treats AMR labels as binary:

- `0` = susceptible
- `1` = resistant
- any other value, including intermediate or missing, is converted to `NaN`.

In [ ]:
# =========================
# Imports
# =========================
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mutual_info_score

try:
    from scipy.cluster.hierarchy import linkage, leaves_list
    from scipy.spatial.distance import squareform
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

: 

In [ ]:
# =========================
# Config
# =========================
DATA_PATH = "/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/data/COMBINED_MARISMA_DRIAMS_samples.pkl"

OUTPUT_DIR = "correlation_analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MIN_PAIR_N = 50          # minimum samples with both antibiotics measured
MIN_AB_N = 50            # minimum samples measured for one antibiotic
MIN_POS_NEG = 2          # minimum number of both classes per antibiotic
TOP_N = 50

In [ ]:
# =========================
# Load dataset
# =========================
with open(DATA_PATH, "rb") as f:
    payload = pickle.load(f)

X_all = payload["data"]
y_species_all = np.asarray(payload["label"])
amr_all = np.asarray(payload["amr"], dtype=float)
antibiotics = np.asarray(payload["antibiotics"])

print("X samples:", len(X_all))
print("AMR matrix shape:", amr_all.shape)
print("Number of antibiotics:", len(antibiotics))
print("Number of species:", len(np.unique(y_species_all)))
print("First antibiotics:", antibiotics[:10])

In [ ]:
# =========================
# Clean AMR matrix
# =========================
def clean_binary_amr(amr):
    """Keep only binary AMR values: 0/1. Everything else becomes NaN."""
    amr = np.asarray(amr, dtype=float).copy()
    valid = (amr == 0) | (amr == 1) | np.isnan(amr)
    amr[~valid] = np.nan
    return amr

amr_bin = clean_binary_amr(amr_all)

unique_raw = pd.Series(amr_all.flatten()).dropna().value_counts().sort_index()
unique_clean = pd.Series(amr_bin.flatten()).dropna().value_counts().sort_index()

print("Raw non-NaN AMR values:")
print(unique_raw)
print("\nCleaned non-NaN AMR values:")
print(unique_clean)

In [ ]:
# =========================
# Basic antibiotic-level statistics
# =========================
def antibiotic_summary(amr, antibiotics):
    rows = []
    for j, ab in enumerate(antibiotics):
        col = amr[:, j]
        valid = ~np.isnan(col)
        n = valid.sum()
        if n == 0:
            n_res = np.nan
            n_sus = np.nan
            resistance_rate = np.nan
        else:
            n_res = int((col[valid] == 1).sum())
            n_sus = int((col[valid] == 0).sum())
            resistance_rate = n_res / n
        rows.append({
            "antibiotic": ab,
            "n_measured": int(n),
            "n_resistant": n_res,
            "n_susceptible": n_sus,
            "resistance_rate": resistance_rate
        })
    return pd.DataFrame(rows).sort_values("n_measured", ascending=False)

summary_global = antibiotic_summary(amr_bin, antibiotics)
summary_global.to_csv(os.path.join(OUTPUT_DIR, "global_antibiotic_summary.csv"), index=False)
summary_global.head(30)

In [ ]:
# =========================
# Filter informative antibiotics
# =========================
def get_informative_antibiotics(amr, min_ab_n=50, min_pos_neg=2):
    valid_cols = []
    for j in range(amr.shape[1]):
        col = amr[:, j]
        vals = col[~np.isnan(col)]
        if len(vals) < min_ab_n:
            continue
        if (vals == 0).sum() < min_pos_neg:
            continue
        if (vals == 1).sum() < min_pos_neg:
            continue
        valid_cols.append(j)
    return np.array(valid_cols, dtype=int)

global_valid_cols = get_informative_antibiotics(amr_bin, MIN_AB_N, MIN_POS_NEG)
print("Informative antibiotics globally:", len(global_valid_cols), "/", amr_bin.shape[1])
print(antibiotics[global_valid_cols])

In [ ]:
# =========================
# Pairwise statistics
# =========================
def phi_corr_binary(x, y):
    """Pearson correlation for two binary vectors; equivalent to phi coefficient."""
    if len(x) < 2:
        return np.nan
    if len(np.unique(x)) < 2 or len(np.unique(y)) < 2:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def odds_ratio_binary(x, y, eps=0.5):
    """Smoothed odds ratio for a 2x2 resistance/susceptibility table."""
    a = np.sum((x == 1) & (y == 1))
    b = np.sum((x == 1) & (y == 0))
    c = np.sum((x == 0) & (y == 1))
    d = np.sum((x == 0) & (y == 0))
    return ((a + eps) * (d + eps)) / ((b + eps) * (c + eps))

def pairwise_resistance_stats(amr, antibiotics, valid_cols=None, min_pair_n=50):
    if valid_cols is None:
        valid_cols = np.arange(amr.shape[1])

    rows = []
    for idx_a in range(len(valid_cols)):
        j = valid_cols[idx_a]
        for idx_b in range(idx_a + 1, len(valid_cols)):
            k = valid_cols[idx_b]

            x = amr[:, j]
            y = amr[:, k]
            valid = ~np.isnan(x) & ~np.isnan(y)
            n = int(valid.sum())
            if n < min_pair_n:
                continue

            x_v = x[valid]
            y_v = y[valid]
            if len(np.unique(x_v)) < 2 or len(np.unique(y_v)) < 2:
                continue

            phi = phi_corr_binary(x_v, y_v)
            mi = mutual_info_score(x_v.astype(int), y_v.astype(int))
            odds_ratio = odds_ratio_binary(x_v, y_v)

            p_b_res_given_a_res = np.mean(y_v[x_v == 1] == 1) if np.sum(x_v == 1) > 0 else np.nan
            p_b_res_given_a_sus = np.mean(y_v[x_v == 0] == 1) if np.sum(x_v == 0) > 0 else np.nan
            p_a_res_given_b_res = np.mean(x_v[y_v == 1] == 1) if np.sum(y_v == 1) > 0 else np.nan
            p_a_res_given_b_sus = np.mean(x_v[y_v == 0] == 1) if np.sum(y_v == 0) > 0 else np.nan

            rows.append({
                "antibiotic_A": antibiotics[j],
                "antibiotic_B": antibiotics[k],
                "idx_A": int(j),
                "idx_B": int(k),
                "n_pair": n,
                "phi_corr": phi,
                "mutual_information": mi,
                "odds_ratio": odds_ratio,
                "P_B_res_given_A_res": p_b_res_given_a_res,
                "P_B_res_given_A_sus": p_b_res_given_a_sus,
                "P_A_res_given_B_res": p_a_res_given_b_res,
                "P_A_res_given_B_sus": p_a_res_given_b_sus,
                "A_res_rate_in_pair": np.mean(x_v == 1),
                "B_res_rate_in_pair": np.mean(y_v == 1),
            })

    df = pd.DataFrame(rows)
    if len(df) > 0:
        df = df.sort_values("phi_corr", ascending=False)
    return df

In [ ]:
# =========================
# Global pairwise correlations
# =========================
global_pairs = pairwise_resistance_stats(
    amr=amr_bin,
    antibiotics=antibiotics,
    valid_cols=global_valid_cols,
    min_pair_n=MIN_PAIR_N
)

global_pairs.to_csv(os.path.join(OUTPUT_DIR, "global_pairwise_resistance_correlations.csv"), index=False)

print("Number of valid global pairs:", len(global_pairs))
global_pairs.head(TOP_N)

In [ ]:
# =========================
# Top positive and negative correlations
# =========================
top_positive = global_pairs.sort_values("phi_corr", ascending=False).head(TOP_N)
top_negative = global_pairs.sort_values("phi_corr", ascending=True).head(TOP_N)

top_positive.to_csv(os.path.join(OUTPUT_DIR, "global_top_positive_correlations.csv"), index=False)
top_negative.to_csv(os.path.join(OUTPUT_DIR, "global_top_negative_correlations.csv"), index=False)

print("Top positive correlations")
display(top_positive[[
    "antibiotic_A", "antibiotic_B", "n_pair", "phi_corr", "mutual_information", "odds_ratio",
    "P_B_res_given_A_res", "P_B_res_given_A_sus"
]])

print("Top negative correlations")
display(top_negative[[
    "antibiotic_A", "antibiotic_B", "n_pair", "phi_corr", "mutual_information", "odds_ratio",
    "P_B_res_given_A_res", "P_B_res_given_A_sus"
]])

In [ ]:
# =========================
# Build matrices from pairwise stats
# =========================
def pairwise_matrix(pair_df, antibiotic_names, value_col="phi_corr", fill_diag=1.0):
    names = list(antibiotic_names)
    mat = pd.DataFrame(np.nan, index=names, columns=names, dtype=float)
    for ab in names:
        mat.loc[ab, ab] = fill_diag
    for _, row in pair_df.iterrows():
        a = row["antibiotic_A"]
        b = row["antibiotic_B"]
        v = row[value_col]
        mat.loc[a, b] = v
        mat.loc[b, a] = v
    return mat

global_phi_matrix = pairwise_matrix(global_pairs, antibiotics[global_valid_cols], value_col="phi_corr")
global_mi_matrix = pairwise_matrix(global_pairs, antibiotics[global_valid_cols], value_col="mutual_information", fill_diag=np.nan)

global_phi_matrix.to_csv(os.path.join(OUTPUT_DIR, "global_phi_matrix.csv"))
global_mi_matrix.to_csv(os.path.join(OUTPUT_DIR, "global_mutual_information_matrix.csv"))

global_phi_matrix.head()

In [ ]:
# =========================
# Heatmap utilities
# =========================
def cluster_order_from_corr(corr_df):
    if not SCIPY_AVAILABLE:
        print("scipy not available; using original order")
        return list(corr_df.index)
    mat = corr_df.fillna(0)
    dist = 1 - np.abs(mat.values)
    np.fill_diagonal(dist, 0)
    condensed = squareform(dist, checks=False)
    Z = linkage(condensed, method="average")
    order = leaves_list(Z)
    return list(mat.index[order])

def plot_heatmap(matrix_df, title, output_path=None, cluster=True, vmin=-1, vmax=1):
    if cluster:
        order = cluster_order_from_corr(matrix_df)
        matrix_df = matrix_df.loc[order, order]

    plt.figure(figsize=(14, 12))
    im = plt.imshow(matrix_df.values, aspect="auto", vmin=vmin, vmax=vmax)
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.xticks(range(len(matrix_df.columns)), matrix_df.columns, rotation=90, fontsize=8)
    plt.yticks(range(len(matrix_df.index)), matrix_df.index, fontsize=8)
    plt.title(title)
    plt.tight_layout()
    if output_path is not None:
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# =========================
# Global correlation heatmap
# =========================
plot_heatmap(
    global_phi_matrix,
    title="Global antibiotic resistance phi correlations",
    output_path=os.path.join(OUTPUT_DIR, "global_phi_heatmap.png"),
    cluster=True,
    vmin=-1,
    vmax=1
)

In [ ]:
# =========================
# Per-species correlation analysis
# =========================
def analyze_species_correlations(species):
    species_mask = (y_species_all == species)
    amr_sp = amr_bin[species_mask]

    valid_cols = get_informative_antibiotics(amr_sp, MIN_AB_N, MIN_POS_NEG)
    if len(valid_cols) < 2:
        return None, None, None

    pairs = pairwise_resistance_stats(
        amr=amr_sp,
        antibiotics=antibiotics,
        valid_cols=valid_cols,
        min_pair_n=MIN_PAIR_N
    )
    if len(pairs) == 0:
        return valid_cols, None, None

    phi_matrix = pairwise_matrix(pairs, antibiotics[valid_cols], value_col="phi_corr")
    return valid_cols, pairs, phi_matrix

species_outputs = {}
all_species_pairs = []

for species in np.unique(y_species_all):
    print("Analyzing", species)
    valid_cols, pairs, phi_matrix = analyze_species_correlations(species)

    if pairs is None:
        print("  skipped: insufficient valid pairs")
        continue

    pairs = pairs.copy()
    pairs.insert(0, "species", species)

    safe_species = str(species).replace("/", "_")
    pairs.to_csv(os.path.join(OUTPUT_DIR, f"{safe_species}_pairwise_correlations.csv"), index=False)
    phi_matrix.to_csv(os.path.join(OUTPUT_DIR, f"{safe_species}_phi_matrix.csv"))

    all_species_pairs.append(pairs)
    species_outputs[species] = {
        "valid_cols": valid_cols,
        "pairs": pairs,
        "phi_matrix": phi_matrix
    }

if all_species_pairs:
    all_species_pairs_df = pd.concat(all_species_pairs, ignore_index=True)
    all_species_pairs_df.to_csv(os.path.join(OUTPUT_DIR, "all_species_pairwise_correlations.csv"), index=False)
else:
    all_species_pairs_df = pd.DataFrame()

print("Done. Species analyzed:", len(species_outputs))

In [ ]:
# =========================
# Strongest positive correlations by species
# =========================
if len(all_species_pairs_df) > 0:
    top_by_species = (
        all_species_pairs_df
        .sort_values(["species", "phi_corr"], ascending=[True, False])
        .groupby("species")
        .head(10)
    )

    top_by_species.to_csv(os.path.join(OUTPUT_DIR, "top_positive_correlations_by_species.csv"), index=False)
    display(top_by_species[[
        "species", "antibiotic_A", "antibiotic_B", "n_pair", "phi_corr",
        "mutual_information", "odds_ratio", "P_B_res_given_A_res", "P_B_res_given_A_sus"
    ]])

In [ ]:
# =========================
# Plot one species heatmap manually
# =========================
# Change this to any species present in species_outputs
species_to_plot = None

# Example:
# species_to_plot = "Escherichia_Coli"

if species_to_plot is not None and species_to_plot in species_outputs:
    safe_species = str(species_to_plot).replace("/", "_")
    plot_heatmap(
        species_outputs[species_to_plot]["phi_matrix"],
        title=f"{species_to_plot} antibiotic resistance phi correlations",
        output_path=os.path.join(OUTPUT_DIR, f"{safe_species}_phi_heatmap.png"),
        cluster=True,
        vmin=-1,
        vmax=1
    )
else:
    print("Set species_to_plot to one of:")
    print(list(species_outputs.keys())[:20])

In [ ]:
# =========================
# Correlation strength distribution
# =========================
plt.figure(figsize=(8, 5))
plt.hist(global_pairs["phi_corr"].dropna(), bins=40)
plt.xlabel("Phi correlation")
plt.ylabel("Number of antibiotic pairs")
plt.title("Distribution of global antibiotic resistance correlations")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "global_phi_distribution.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# =========================
# Candidate edges for graph/contextual inference
# =========================
# These strong positive relationships can later be used as:
# - graph edges between antibiotics
# - biologically-informed autoregressive ordering
# - priors for confidence propagation

EDGE_PHI_THRESHOLD = 0.30
EDGE_MIN_PAIR_N = 100

candidate_edges = global_pairs[
    (global_pairs["phi_corr"] >= EDGE_PHI_THRESHOLD) &
    (global_pairs["n_pair"] >= EDGE_MIN_PAIR_N)
].copy()

candidate_edges = candidate_edges.sort_values("phi_corr", ascending=False)
candidate_edges.to_csv(os.path.join(OUTPUT_DIR, "candidate_antibiotic_correlation_edges.csv"), index=False)

print("Candidate edges:", len(candidate_edges))
display(candidate_edges[[
    "antibiotic_A", "antibiotic_B", "n_pair", "phi_corr", "mutual_information",
    "odds_ratio", "P_B_res_given_A_res", "P_B_res_given_A_sus"
]].head(100))

In [ ]:
# =========================
# Optional adjacency matrix for future graph model
# =========================
adj = pd.DataFrame(
    np.zeros((len(global_valid_cols), len(global_valid_cols))),
    index=antibiotics[global_valid_cols],
    columns=antibiotics[global_valid_cols]
)

for _, row in candidate_edges.iterrows():
    a = row["antibiotic_A"]
    b = row["antibiotic_B"]
    w = row["phi_corr"]
    adj.loc[a, b] = w
    adj.loc[b, a] = w

adj.to_csv(os.path.join(OUTPUT_DIR, "candidate_antibiotic_graph_adjacency.csv"))
adj.head()

In [ ]:
# =========================
# Final output inventory
# =========================
print("Files saved in:", OUTPUT_DIR)
for fname in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", fname)